# BERT/RoBERTa Model Evaluation Pipeline

For the models to be considered for implementation, it will undergo comparitive evaluation against each other. Models that achieve an F1-Scire of 0.75 or above will be considered, with F1 Score being the secondary factor. Additionally, a confusion matrix is created to visualise the performance of the models.

Chosen Models for Evaluation:
1. hamzab/roberta-fake-news-classification
2. jy46604790/Fake-News-Bert-Detect
3. mrm8488/bert-tiny-finetuned-fake-news-detection
4. elozano/bert-base-cased-fake-news
5. Arko007/fake-news-roberta-5M

In [45]:
!pip install torch numpy pandas tqdm transformers scikit-learn

In [46]:
# Importing Dependencies
import os
import gc
import json
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, confusion_matrix

# Variable Definitions

In [67]:
# Dataset CSV Path
csv_path = '../isot_bert_test.csv'

# Results Directory
results_dir = './bert_eval_results'

# Text Data Column
text_col = 'combined'

# Label Data Column
label_col = 'label'

# Models
models = [
    "./models/hamzab-roberta-fake-news-classification",
    "./models/jy46604790-Fake-News-Bert-Detect",
    "./models/mrm8488-bert-tiny-finetuned-fake-news-detection",
    "./models/elozano-bert-base-cased-fake-news",
    "./models/Arko007-fake-news-roberta-5M",
]

# Utilizing CPU due to issues with CUDA
device = "cpu"
print(f"Running on: {device.upper()}")

Running on: CPU


# Helper Functions

## Compute Metrics

Checks for any empty result values, before calculating and returning the F1 Score, Valid and Invalid samples and the Confusion Matrix.

In [68]:
def compute_metrics(y_true, y_pred):
    # Filtering out none values in case
    valid_mask = [p is not None for p in y_pred]
    valid_y_true = [y for y, m in zip(y_true, valid_mask) if m]
    valid_preds = [p for p, m in zip(y_pred, valid_mask) if m]
    valid_count = sum(valid_mask)
    invalid_count = len(y_pred) - valid_count

    # If no valid values (all none)
    if not valid_preds:
        return none

    y_true_arr = np.array(valid_y_true)
    preds_arr = np.array(valid_preds)

    # F1-Score
    f1 = f1_score(y_true_arr, preds_arr)

    # Confusion Matrix at 0.5 threshold
    cm = confusion_matrix(y_true_arr, preds_arr)
    tn, fp, fn, tp = cm.ravel()

    return {
        "f1_score": round(f1, 4),
        "valid_samples": valid_count,
        "invalid_samples": invalid_count,
        "confusion_matrix": {"tn": int(tn), "fp": int(fp),
                             "fn": int(fn), "tp": int(tp)},
    }

## Print Confusion Matrix

Helper function to prin the results of the confusion matrix

In [69]:
def print_confusion_matrix(cm_dict, model_name):
    # Printing the confusion matrix in console
    print(f"\nConfusion Matrix — {model_name}")
    print(f"                 Predicted Real  Predicted Fake")
    print(f"  Actual Real       {cm_dict['tn']}              {cm_dict['fp']}")
    print(f"  Actual Fake       {cm_dict['fn']}              {cm_dict['tp']}")

## Loading in the dataset

In [70]:
# Loading in the dataset
df = pd.read_csv(csv_path)
y_true = df[label_col].tolist()

# Master DataFrame to store per-sample scores across all models
df_results = df[[label_col]].copy()
df_results['binary_label'] = y_true

# Creating the directory for results
os.makedirs(results_dir, exist_ok = True)
summary_rows = []

# Multi-Model Inference Loop

This for loop runs an iterative inference test on each model selected. The results are then collated, calculated and displayed using the helper functions above.

In [71]:
for model_path in models:
    # Retrieving model name for logs
    model_name = os.path.basename(model_path.rstrip('/'))

    print(f"Loading Model: {model_name}...")

    # Loading Model In
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.eval()
    
    y_pred = []
    
    # Running inference (no gradient racking for memory and speed)
    with torch.no_grad():

        for index, row in tqdm(df.iterrows(), total = len(df), desc = model_name):

            text = row[text_col]

            try:

                # Tokenizer handles the tokinzing of text
                inputs = tokenizer(text,
                                   return_tensors = "pt",
                                   truncation = True,
                                   max_length = 512,
                                   padding = True).to(device)

                outputs = model(**inputs)
                pred = torch.argmax(outputs.logits, dim = 1).item()

                y_pred.append(pred)

            except Exception as e:
                print(e)
                y_pred.append(None)
        
        # Saving per-sample model scores to results DF
        df_results[f"{model_name}_pred"] = y_pred
        df_results.to_csv(os.path.join(results_dir, "interim_scores.csv"), index = False)

        # Compute and display metrics
        metrics = compute_metrics(y_true, y_pred)

        # Illustrating the metrics
        if metrics:
            cm = metrics['confusion_matrix']
            print(f"F1 Score: {metrics['f1_score']}")
            print(f"Valid samples: {metrics['valid_samples']}")
            print(f"Invalid samples: {metrics['invalid_samples']}")
            print_confusion_matrix(cm, model_name)
    
            # Save per-model metrics as JSON
            json_path = os.path.join(results_dir, f"{model_name}_metrics.json")
            with open(json_path, 'w') as f:
                json.dump(metrics, f, indent = 2)

            # Saving a summarised result table
            summary_rows.append({
                "model": model_name,
                "f1_score": metrics["f1_score"],
                "valid_samples": metrics["valid_samples"],
                "invalid_samples": metrics["invalid_samples"],
                "tp": cm["tp"], "fp": cm["fp"],
                "fn": cm["fn"], "tn": cm["tn"],
            })
        else:
            print(f"{model_name} produced no valid results.")

        
        # Wipe the local memory clean before the loop restarts for the next model
        print(f"\nClearing {model_name} from local memory...")
        del model
        del tokenizer
        if 'inputs' in locals(): del inputs
        if 'outputs' in locals(): del outputs
        gc.collect()
        print(f"\nCache Cleared.")

Loading Model: hamzab-roberta-fake-news-classification...


hamzab-roberta-fake-news-classification: 100%|█████████████████████████████████████| 2000/2000 [04:38<00:00,  7.19it/s]


F1 Score: 1.0
Valid samples: 2000
Invalid samples: 0

Confusion Matrix — hamzab-roberta-fake-news-classification
                 Predicted Real  Predicted Fake
  Actual Real       1031              0
  Actual Fake       0              969

Clearing hamzab-roberta-fake-news-classification from local memory...

Cache Cleared.
Loading Model: jy46604790-Fake-News-Bert-Detect...


jy46604790-Fake-News-Bert-Detect: 100%|████████████████████████████████████████████| 2000/2000 [04:29<00:00,  7.43it/s]


F1 Score: 1.0
Valid samples: 2000
Invalid samples: 0

Confusion Matrix — jy46604790-Fake-News-Bert-Detect
                 Predicted Real  Predicted Fake
  Actual Real       1031              0
  Actual Fake       0              969

Clearing jy46604790-Fake-News-Bert-Detect from local memory...

Cache Cleared.
Loading Model: mrm8488-bert-tiny-finetuned-fake-news-detection...


...

F1 Score: 0.0014
Valid samples: 2000
Invalid samples: 0

Confusion Matrix — mrm8488-bert-tiny-finetuned-fake-news-detection
                 Predicted Real  Predicted Fake
  Actual Real       529              502
  Actual Fake       968              1

Clearing mrm8488-bert-tiny-finetuned-fake-news-detection from local memory...

Cache Cleared.
Loading Model: elozano-bert-base-cased-fake-news...


elozano-bert-base-cased-fake-news: 100%|███████████████████████████████████████████| 2000/2000 [04:30<00:00,  7.40it/s]


F1 Score: 0.9995
Valid samples: 2000
Invalid samples: 0

Confusion Matrix — elozano-bert-base-cased-fake-news
                 Predicted Real  Predicted Fake
  Actual Real       1031              0
  Actual Fake       1              968

Clearing elozano-bert-base-cased-fake-news from local memory...

Cache Cleared.
Loading Model: Arko007-fake-news-roberta-5M...


Arko007-fake-news-roberta-5M: 100%|████████████████████████████████████████████████| 2000/2000 [04:28<00:00,  7.45it/s]


F1 Score: 0.9964
Valid samples: 2000
Invalid samples: 0

Confusion Matrix — Arko007-fake-news-roberta-5M
                 Predicted Real  Predicted Fake
  Actual Real       1024              7
  Actual Fake       0              969

Clearing Arko007-fake-news-roberta-5M from local memory...

Cache Cleared.


# Printing of Results

In [72]:
# Reading and illustrating the summary results for all models together
if summary_rows:

    # Sorting and saving the summary results as dataframe and csv
    df_summary = pd.DataFrame(summary_rows).sort_values("f1_score", ascending = False)
    summary_path = os.path.join(results_dir, "bert_model_comparison.csv")
    df_summary.to_csv(summary_path, index = False)

    # Printing the comparison
    print(f"\n{'='*60}")
    print("Model Comparison by F! Score")
    print(f"{'='*60}")
    print(df_summary[["model", "f1_score"]].to_string(index = False))

# Save all per-sample scores for future reference
df_results.to_csv(os.path.join(results_dir, "bert_all_scores.csv"), index = False)
print(f"\nEvaluation complete. All results are stored in: {results_dir}/")


Model Comparison by F! Score
                                          model  f1_score
        hamzab-roberta-fake-news-classification    1.0000
               jy46604790-Fake-News-Bert-Detect    1.0000
              elozano-bert-base-cased-fake-news    0.9995
                   Arko007-fake-news-roberta-5M    0.9964
mrm8488-bert-tiny-finetuned-fake-news-detection    0.0014

Evaluation complete. All results are stored in: ./bert_eval_results/


By comparing the rersults above, it can be noted that the Hamzab roberta and JY46604790 Bert models are both capable of perfect results on the test dataset. To select between the two, I will select the which contains less parameters, resulting in faster inference speeds.

In [73]:
# Hamzab
model = AutoModelForSequenceClassification.from_pretrained(
    "./models/hamzab-roberta-fake-news-classification")
print(sum(p.numel() for p in model.parameters()))

Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 6481.55it/s]

124647170


In [74]:
# JY
model = AutoModelForSequenceClassification.from_pretrained(
    "./models/jy46604790-Fake-News-Bert-Detect")
print(sum(p.numel() for p in model.parameters()))

Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 6657.99it/s]

124647170
